In [9]:
# read C:\Users\ducli\Downloads\archive (1)\vietnam_housing_dataset.csv
import pandas as pd

df = pd.read_csv(r'vietnam_housing_dataset.csv')
df.head()

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,NaN,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,NaN,NaN,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90


In [10]:
df.drop(columns=['Address', 'House direction', 'Balcony direction'], inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [11]:
df['Have certificate'] = df['Legal status'].map({'Have certificate': 1, 'Sale contract': 0})
df['Full furniture state'] = df['Furniture state'].map({'Full': 1, 'Basic': 0})
df.drop(columns=['Legal status', 'Furniture state'], inplace=True)
price_col = df.pop('Price')
df['Price'] = price_col
df.head()

,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Have certificate,Full furniture state,Price
16,300.0,15.0,32.0,3.0,6.0,3.0,1,1,9.40
19,46.0,4.6,6.0,4.0,4.0,5.0,1,1,7.99
23,60.0,3.5,5.0,2.0,6.0,5.0,1,1,5.60
24,369.0,15.0,12.0,2.0,4.0,4.0,1,1,6.30
28,67.4,4.0,8.0,5.0,4.0,6.0,1,0,9.50


In [12]:
import math
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [13]:
X = df.drop(columns=["Price"])
y = df["Price"]
#X.head()
#y.head()

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#X_train.head()

In [19]:
models = {
    "Linear-Regression": LinearRegression(),
    "Polynomial-Regression(deg=2)": make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False), LinearRegression()
    ),
    "Random-Forest(n_estimators=300)": RandomForestRegressor(n_estimators=300, random_state=42),
}
metrics = {}


In [24]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 5. Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    metrics[name] = {
        "MAE (tỷ VNĐ)": mae,
        "RMSE (tỷ VNĐ)": rmse,
        "MAPE (%)": mape,
        "R²": r2,
        
    }

In [25]:
results_df = pd.DataFrame(metrics).T.round(3)

In [26]:
results_df

,MAE (tỷ VNĐ),RMSE (tỷ VNĐ),MAPE (%),R²
Linear-Regression,1.509,1.855,34.112,0.294
Polynomial-Regression(deg=2),1.440,1.780,32.310,0.350
Random-Forest(n_estimators=300),1.354,1.720,29.696,0.394


In [16]:
X_train.describe()

,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Have certificate,Full furniture state
count,6258.000000,6258.000000,6258.000000,6258.000000,6258.000000,6258.000000,6258.000000,6258.000000
mean,64.711569,5.340900,6.719655,3.361937,3.533078,3.427772,0.984180,0.647172
std,42.831487,4.885514,6.161093,1.334750,1.235150,1.353220,0.124788,0.477888
min,3.100000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,40.000000,4.000000,3.000000,2.000000,3.000000,2.000000,1.000000,0.000000
50%,55.000000,4.200000,5.000000,3.000000,3.000000,3.000000,1.000000,1.000000
75%,76.000000,5.000000,7.500000,4.000000,4.000000,4.000000,1.000000,1.000000
max,524.700000,75.000000,85.000000,7.000000,9.000000,9.000000,1.000000,1.000000
